# build_silver_kev
`stg_kev_valid`(이벤트 단위: `event_type` + `cve_id` + `catalog_version` + `detected_at` + `source_commit_sha` + `data`) → `silver_kev` (Change History, grain: `cve_id` × valid period).

**더 이상 "어제 KEV 전체 vs 오늘 KEV 전체" 스냅샷 diff를 하지 않는다.** 입력 자체가 이미 무엇이 바뀌었는지 `event_type`으로 알려주므로, 이 노트북은 다음 순서만 수행한다.

```
Bronze Event 읽기(stg_kev_valid)
  → event_type 확인
  → data 정규화
  → 기존 silver_kev 상태 확인
  → CORE_FIELDS 기준 의미 있는 변화인지 판단
  → valid_from / valid_to 갱신
```

- **`kev_added` / `kev_updated`**: 기존 SCD2 해시 비교 로직 그대로 적용 — 신규 키면 새로 열고, `CORE_FIELDS` 해시가 바뀌었으면 기존 버전을 닫고 새로 열고, 안 바뀌었으면 서술형 필드 + `last_seen_at`만 갱신한다.
- **`kev_removed_confirmed`**: 현재 열린 버전을 **닫기만** 한다(`valid_to = detected_at`). 새 버전은 열지 않는다 — `valid_to IS NULL`인 행이 없다는 것 자체가 "현재 KEV 미등재"를 의미하므로, `kev_listed=True`가 "이 테이블에 있다는 것 자체가 등재 확인"이라는 기존 원칙이 자연스럽게 이어진다.
- **`valid_from`은 이제 이벤트의 `detected_at`(실제 발생 시각)을 최우선으로 쓴다** — Function이 이미 정확한 변경 시점을 알려주므로, 노트북 실행 시각(`ingested_at`)보다 신뢰도가 높다.
- **개선 방향 3가지, 그대로 유지** (02_Bronze_Silver_저장구조.md 💡):
  1. `record_hash`는 `CORE_FIELDS = [cve_id, date_added, due_date, known_ransomware_use, kev_listed]`만 반영.
  2. `dq_flags`는 파생 필드라 해시 대상에서 제외.
  3. 해시가 같으면(`kev_updated`로 왔어도 핵심값은 안 바뀐 경우) 새 버전을 안 열고 서술형 필드 + `last_seen_at`만 갱신.
- **`catalog_version`은 lineage 참고용으로만 보관한다.** `kev_removed_confirmed`는 새 fetch 없이도 생길 수 있어 `catalog_version`이 이벤트 발생 시점의 최신 Catalog를 항상 의미하지 않을 수 있다 — 그래서 별도 `STALE_CATALOG_VERSION` 같은 DQ 플래그는 만들지 않고, 상태 판정 기준으로도 쓰지 않는다.

In [ ]:
%run 00_silver_common

In [ ]:
from pyspark.sql import Window

raw_df = spark.table(TBL["stg_kev_valid"])

# 같은 batch(워터마크 구간) 안에 동일 cve_id 이벤트가 여러 개 섞여 있으면,
# detected_at 기준 가장 최신 이벤트 하나로 그 CVE의 최종 상태를 결정한다.
w = Window.partitionBy("cve_id_raw").orderBy(F.col("detected_at").desc())
latest_events = (
    raw_df
    .withColumn("_rn", F.row_number().over(w))
    .filter(F.col("_rn") == 1)
    .drop("_rn")
)


In [ ]:
cve_id = F.col("cve_id_raw")
vendor_project = nested_coalesce(latest_events, ["data.vendorProject", "data.vendor_project"])
product = nested_coalesce(latest_events, ["data.product"])
vulnerability_name = nested_coalesce(latest_events, ["data.vulnerabilityName", "data.vulnerability_name"])
date_added = nested_coalesce(latest_events, ["data.dateAdded", "data.date_added"])
short_description = nested_coalesce(latest_events, ["data.shortDescription", "data.short_description"])
required_action = nested_coalesce(latest_events, ["data.requiredAction", "data.required_action"])
due_date = nested_coalesce(latest_events, ["data.dueDate", "data.due_date"])
known_ransomware_use = nested_coalesce(
    latest_events, ["data.knownRansomwareCampaignUse", "data.known_ransomware_campaign_use"]
)
cwes = F.coalesce(nested_or_none(latest_events, "data.cwes"), F.array().cast("array<string>"))
notes = nested_coalesce(latest_events, ["data.notes"])
catalog_version = existing_col(latest_events, "catalog_version")        # lineage 전용
source_commit_sha = existing_col(latest_events, "source_commit_sha")    # lineage 전용

dq_flags = flags_array(
    F.when(date_added.isNull(), F.lit("MISSING_DATE_ADDED")),
    F.when(due_date.isNull(), F.lit("MISSING_DUE_DATE")),
)
valid_from = F.coalesce(F.col("detected_at"), date_added, F.lit(INGESTED_AT))


In [ ]:
CORE_FIELDS = [cve_id, date_added, due_date, known_ransomware_use, F.lit(True)]

event_rows = latest_events.select(
    cve_id.alias("cve_id"),
    F.col("event_type"),
    vendor_project.alias("vendor_project"),
    product.alias("product"),
    vulnerability_name.alias("vulnerability_name"),
    date_added.alias("date_added"),
    short_description.alias("short_description"),
    required_action.alias("required_action"),
    due_date.alias("due_date"),
    known_ransomware_use.alias("known_ransomware_use"),
    cwes.alias("cwes"),
    notes.alias("notes"),
    catalog_version.alias("catalog_version"),
    source_commit_sha.alias("source_commit_sha"),
    F.lit(True).alias("kev_listed"),
    dq_flags.alias("dq_flags"),
    valid_from.alias("valid_from"),
    F.col("detected_at"),
    F.lit("cisa_kev").alias("source"),
    F.col("batch_id"),
    F.col("ingested_at"),
).withColumn("record_hash", content_hash(CORE_FIELDS))

# kev_added/kev_updated -> 기존 SCD2 해시 비교 경로. kev_removed_confirmed -> 버전 닫기 전용 경로.
upsert_events = event_rows.filter(F.col("event_type").isin(["kev_added", "kev_updated"])).drop("event_type")
removed_events = event_rows.filter(F.col("event_type") == "kev_removed_confirmed").drop("event_type")

upsert_count = upsert_events.count()
removed_count = removed_events.count()


In [ ]:
DESC_UPDATE_SET = {
    "vendor_project": "s.vendor_project",
    "product": "s.product",
    "vulnerability_name": "s.vulnerability_name",
    "short_description": "s.short_description",
    "required_action": "s.required_action",
    "notes": "s.notes",
    "cwes": "s.cwes",
    "catalog_version": "s.catalog_version",
    "source_commit_sha": "s.source_commit_sha",
    "dq_flags": "s.dq_flags",
    "last_seen_at": "s.ingested_at",
    "batch_id": "s.batch_id",
}

opened_count = changed_count = unchanged_count = 0

if upsert_count > 0:
    if not table_exists(TBL["kev"]):
        first_version = (
            upsert_events
            .withColumn("valid_to", F.lit(None).cast(StringType()))
            .withColumn("last_seen_at", F.col("ingested_at"))
        )
        write_overwrite(first_version, TBL["kev"])
        opened_count = upsert_count
    else:
        target = DeltaTable.forName(spark, TBL["kev"])
        current_open = target.toDF().filter(F.col("valid_to").isNull())

        compared = (
            current_open.alias("cur")
            .join(upsert_events.alias("new"), current_open["cve_id"] == upsert_events["cve_id"], "inner")
            .select(
                F.col("new.cve_id").alias("cve_id"),
                F.col("cur.record_hash").alias("cur_hash"),
                F.col("new.record_hash").alias("new_hash"),
                F.col("new.valid_from").alias("new_valid_from"),
            )
        )
        changed = compared.filter(F.col("cur_hash") != F.col("new_hash"))
        unchanged = compared.filter(F.col("cur_hash") == F.col("new_hash"))
        changed_count = changed.count()
        unchanged_count = unchanged.count()

        if changed_count > 0:
            changed.select("cve_id", F.col("new_valid_from").alias("close_at")) \
                .createOrReplaceTempView("_kev_close")
            (
                target.alias("t")
                .merge(spark.table("_kev_close").alias("s"), "t.cve_id = s.cve_id AND t.valid_to IS NULL")
                .whenMatchedUpdate(set={"valid_to": "s.close_at"})
                .execute()
            )

        if unchanged_count > 0:
            desc_updates = upsert_events.join(unchanged.select("cve_id"), on="cve_id", how="inner")
            desc_updates.createOrReplaceTempView("_kev_desc_updates")
            (
                target.alias("t")
                .merge(spark.table("_kev_desc_updates").alias("s"), "t.cve_id = s.cve_id AND t.valid_to IS NULL")
                .whenMatchedUpdate(set=DESC_UPDATE_SET)
                .execute()
            )

        cur_hash_lookup = current_open.select("cve_id", F.col("record_hash").alias("cur_hash"))
        new_or_changed = (
            upsert_events.join(cur_hash_lookup, on="cve_id", how="left")
            .filter(F.col("cur_hash").isNull() | (F.col("cur_hash") != F.col("record_hash")))
            .drop("cur_hash")
            .withColumn("valid_to", F.lit(None).cast(StringType()))
            .withColumn("last_seen_at", F.col("ingested_at"))
        )
        opened_count = new_or_changed.count()
        if opened_count > 0:
            write_append(new_or_changed, TBL["kev"])


In [ ]:
# kev_removed_confirmed: 현재 열린 버전을 닫기만 한다 -- 새 버전을 열지 않는다.
if removed_count > 0 and table_exists(TBL["kev"]):
    removed_events.select("cve_id", "detected_at", "batch_id", "source_commit_sha") \
        .createOrReplaceTempView("_kev_removed")
    target = DeltaTable.forName(spark, TBL["kev"])
    (
        target.alias("t")
        .merge(spark.table("_kev_removed").alias("s"), "t.cve_id = s.cve_id AND t.valid_to IS NULL")
        .whenMatchedUpdate(set={
            "valid_to": "s.detected_at",
            "batch_id": "s.batch_id",
            "source_commit_sha": "s.source_commit_sha",
        })
        .execute()
    )
    # silver_kev에 이미 열린 버전이 없는 CVE에 removed 이벤트만 들어온 경우(순서 역전 등)는
    # 매칭될 행이 없어 자연히 아무 것도 하지 않는다 -- 존재하지 않는 상태를 굳이 만들지 않는다.

log_batch_run("build_silver_kev", "SUCCESS", record_count=upsert_count + removed_count)
log_dq_metric(
    "build_silver_kev", "versions_opened", opened_count, passed=True,
    detail=f"changed={changed_count}, unchanged_desc_only={unchanged_count}, removed_closed={removed_count}",
)
print(f"[build_silver_kev] opened={opened_count} changed={changed_count} "
      f"unchanged_desc_only={unchanged_count} removed_closed={removed_count} batch_id(run)={batch_id}")
